# Nettoyage et qualité des données Instacart

Ce notebook regroupe l?exploration rapide, le nettoyage prudent, les tests de qualit? et le rapport pour les six fichiers Instacart. Il est autonome : les fonctions utilis?es sont d?finies dans les cellules ci-dessous.

Les sources de `data/raw/` ne sont jamais modifi?es. Les copies nettoy?es et le rapport sont ?crits dans `data/processed/`.

## 1. Préparer l'environnement

Ex?cuter les cellules dans l'ordre. Le dossier du projet est repéré à partir de `data/raw/`.

In [2]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

# Le notebook est pr?vu pour ?tre lanc? depuis le dossier du projet ou un de ses sous-dossiers.
candidats = [Path.cwd(), *Path.cwd().parents]
PROJET = next(
    (p for p in candidats if (p / "data" / "raw").is_dir()),
    None,
)
if PROJET is None:
    raise FileNotFoundError("Racine du projet introuvable : le dossier data/raw est absent.")

DOSSIER_RAW = PROJET / "data" / "raw"
DOSSIER_PREPARE = PROJET / "data" / "processed"
DOSSIER_RAPPORTS = DOSSIER_PREPARE
print(f"Projet : {PROJET}")
print(f"Donn?es brutes : {DOSSIER_RAW}")

Projet : c:\Users\mouft\Desktop\data\data_m2
Donn?es brutes : c:\Users\mouft\Desktop\data\data_m2\data\raw


## 2. Vérifier les fichiers et regarder un apercu

On lit seulement les cinq premieres lignes de chaque CSV pour découvrir les colonnes sans charger les fichiers volumineux en mémoire.

In [3]:
FICHIERS = [
    "aisles.csv",
    "departments.csv",
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
]

for nom in FICHIERS:
    chemin = DOSSIER_RAW / nom
    if not chemin.is_file():
        raise FileNotFoundError(f"Fichier brut manquant : {chemin}")
    apercu = pd.read_csv(chemin, nrows=5)
    print(f"\n{nom} ? {chemin.stat().st_size / 1_000_000:.1f} Mo")
    display(apercu)


aisles.csv ? 0.0 Mo


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation



departments.csv ? 0.0 Mo


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol



orders.csv ? 109.0 Mo


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


... (2 outputs left out)


order_products__train.csv ? 24.7 Mo


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1



products.csv ? 2.2 Mo


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## 3. Définir les règles de qualité

Les règles ci-dessous décrivent les colonnes attendues, les identifiants qui doivent ètre uniques et les plages de valeurs acceptables. Le script lit les gros CSV par morceaux de 250 000 lignes pour limiter l'usage mémoire.

In [9]:
TAILLE_MORCEAU = 250_000

SCHEMAS = {
    "aisles.csv": {
        "colonnes": ["aisle_id", "aisle"],
        "cle": ["aisle_id"],
        "textes": ["aisle"],
        "plages": {"aisle_id": (1, None)},
    },
    "departments.csv": {
        "colonnes": ["department_id", "department"],
        "cle": ["department_id"],
        "textes": ["department"],
        "plages": {"department_id": (1, None)},
    },
    "products.csv": {
        "colonnes": ["product_id", "product_name", "aisle_id", "department_id"],
        "cle": ["product_id"],
        "textes": ["product_name"],
        "plages": {
            "product_id": (1, None),
            "aisle_id": (1, None),
            "department_id": (1, None),
        },
    },
    "orders.csv": {
        "colonnes": [
            "order_id",
            "user_id",
            "eval_set",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order",
        ],
        "cle": ["order_id"],
        "textes": ["eval_set"],
        "plages": {
            "order_id": (1, None),
            "user_id": (1, None),
            "order_number": (1, None),
            "order_dow": (0, 6),
            "order_hour_of_day": (0, 23),
            "days_since_prior_order": (0, 30),
        },
    },
    "order_products__prior.csv": {
        "colonnes": ["order_id", "product_id", "add_to_cart_order", "reordered"],
        "cle": ["order_id", "product_id"],
        "textes": [],
        "plages": {
            "order_id": (1, None),
            "product_id": (1, None),
            "add_to_cart_order": (1, None),
            "reordered": (0, 1),
        },
    },
    "order_products__train.csv": {
        "colonnes": ["order_id", "product_id", "add_to_cart_order", "reordered"],
        "cle": ["order_id", "product_id"],
        "textes": [],
        "plages": {
            "order_id": (1, None),
            "product_id": (1, None),
            "add_to_cart_order": (1, None),
            "reordered": (0, 1),
        },
    },
}

def ajouter_test(tests, nom, reussi, detail, avertissement=False):
    if reussi:
        statut = "OK"
    elif avertissement:
        statut = "AVERTISSEMENT"
    else:
        statut = "ÉCHEC"
    tests.append(f"[{statut}] {nom} : {detail}")

def tableau_identifiants(valeurs):
    if not valeurs:
        return np.array([], dtype=np.int64)
    identifiants = np.concatenate(valeurs)
    identifiants = identifiants[~pd.isna(identifiants)]
    return np.sort(np.unique(identifiants.astype(np.int64)))

def compter_references_absentes(valeurs, references):
    valeurs = pd.to_numeric(valeurs, errors="coerce").to_numpy()
    presentes = ~pd.isna(valeurs)
    valeurs = valeurs[presentes].astype(np.int64)

    if len(valeurs) == 0:
        return 0
    if len(references) == 0:
        return len(valeurs)

    positions = np.searchsorted(references, valeurs)
    dans_plage = positions < len(references)
    correspondances = np.zeros(len(valeurs), dtype=bool)
    correspondances[dans_plage] = (
        references[positions[dans_plage]] == valeurs[dans_plage]
    )
    return int((~correspondances).sum())

## 4. Analyser, corriger prudemment et tester

La fonction suivante v?rifie les valeurs manquantes, les cl?s, les plages et les r?f?rences entre tables. La seule correction automatique est de retirer les espaces superflus des textes. Elle ne supprime aucune ligne et n?invente aucune valeur.

In [5]:
def analyser_fichier(nom_fichier, references, tests):
    chemin = DOSSIER_RAW / nom_fichier
    chemin_prepare = DOSSIER_PREPARE / nom_fichier
    schema = SCHEMAS[nom_fichier]

    colonnes_disponibles = pd.read_csv(chemin, nrows=0).columns.tolist()
    colonnes_manquantes = sorted(
        set(schema["colonnes"]) - set(colonnes_disponibles)
    )
    if colonnes_manquantes:
        raise ValueError(
            f"{nom_fichier} : colonnes absentes : {colonnes_manquantes}"
        )

    lignes = 0
    manquants = {colonne: 0 for colonne in schema["colonnes"]}
    hors_plage = {colonne: 0 for colonne in schema["plages"]}
    valeurs_invalides = {colonne: 0 for colonne in schema["plages"]}
    corrections_espaces = {colonne: 0 for colonne in schema["textes"]}
    corrections_vides = {colonne: 0 for colonne in schema["textes"]}
    identifiants = []
    doublons_cle = 0
    ordre_trie = True
    derniere_commande = None
    produits_derniere_commande = set()
    commandes_premieres_sans_delai = 0
    commandes_non_premieres_sans_delai = 0
    delai_sur_premiere_commande = 0
    references_absentes = {nom: 0 for nom in references}

    chemin_prepare.parent.mkdir(parents=True, exist_ok=True)

    for numero_morceau, morceau in enumerate(
        pd.read_csv(chemin, chunksize=TAILLE_MORCEAU, low_memory=False)
    ):
        lignes += len(morceau)
        for colonne in schema["colonnes"]:
            manquants[colonne] += int(morceau[colonne].isna().sum())

        for colonne, (minimum, maximum) in schema["plages"].items():
            numerique = pd.to_numeric(morceau[colonne], errors="coerce")
            valeurs_invalides[colonne] += int(
                (morceau[colonne].notna() & numerique.isna()).sum()
            )
            invalides = morceau[colonne].notna() & numerique.isna()
            if minimum is not None:
                invalides |= numerique.notna() & (numerique < minimum)
            if maximum is not None:
                invalides |= numerique.notna() & (numerique > maximum)
            hors_plage[colonne] += int(invalides.sum())

        if len(schema["cle"]) == 1:
            colonne_cle = schema["cle"][0]
            identifiants.append(
                pd.to_numeric(morceau[colonne_cle], errors="coerce")
                .dropna()
                .to_numpy()
            )

        if schema["cle"] == ["order_id", "product_id"]:
            premiere_id = morceau["order_id"].iloc[0] if len(morceau) else None
            derniere_id = morceau["order_id"].iloc[-1] if len(morceau) else None
            ordre_trie = ordre_trie and morceau["order_id"].is_monotonic_increasing
            if (
                derniere_commande is not None
                and premiere_id is not None
                and premiere_id < derniere_commande
            ):
                ordre_trie = False

            doublons_cle += int(morceau.duplicated(subset=schema["cle"]).sum())

            if len(morceau):
                premiere_lignes = morceau[
                    morceau["order_id"] == premiere_id
                ]["product_id"]
                produits_premiere_commande = set(premiere_lignes.unique())
                if premiere_id == derniere_commande:
                    doublons_cle += len(
                        produits_premiere_commande
                        & produits_derniere_commande
                    )

                derniere_lignes = morceau[
                    morceau["order_id"] == derniere_id
                ]["product_id"]
                produits_derniere_commande_morceau = set(
                    derniere_lignes.unique()
                )
                if derniere_id == derniere_commande:
                    produits_derniere_commande.update(
                        produits_derniere_commande_morceau
                    )
                else:
                    produits_derniere_commande = (
                        produits_derniere_commande_morceau
                    )
            derniere_commande = derniere_id

        if nom_fichier == "orders.csv":
            premier = morceau["order_number"].eq(1)
            delai_manquant = morceau["days_since_prior_order"].isna()
            commandes_premieres_sans_delai += int((premier & delai_manquant).sum())
            commandes_non_premieres_sans_delai += int(
                (~premier & delai_manquant).sum()
            )
            delai_sur_premiere_commande += int(
                (premier & ~delai_manquant).sum()
            )

        for nom_reference, colonnes_reference in references.items():
            colonne = {
                "aisle_id": "aisle_id",
                "department_id": "department_id",
                "order_id": "order_id",
                "product_id": "product_id",
            }[nom_reference]
            if colonne in morceau.columns:
                references_absentes[nom_reference] += compter_references_absentes(
                    morceau[colonne], colonnes_reference
                )

        for colonne in schema["textes"]:
            texte = morceau[colonne]
            espaces = texte.map(
                lambda valeur: isinstance(valeur, str)
                and valeur != valeur.strip()
            )
            vides = texte.map(
                lambda valeur: isinstance(valeur, str)
                and valeur.strip() == ""
            )
            corrections_espaces[colonne] += int(espaces.sum())
            corrections_vides[colonne] += int(vides.sum())
            morceau[colonne] = texte.map(
                lambda valeur: (
                    valeur.strip() if isinstance(valeur, str) else valeur
                )
            )
            morceau.loc[morceau[colonne].eq(""), colonne] = pd.NA

        if (numero_morceau + 1) % 10 == 0:
            print(
                f"{nom_fichier} : {lignes:,} lignes analysées..."
            )

        morceau.to_csv(
            chemin_prepare,
            index=False,
            mode="w" if numero_morceau == 0 else "a",
            header=numero_morceau == 0,
        )

    ids_uniques = tableau_identifiants(identifiants)
    if len(schema["cle"]) == 1:
        valeurs_cle_valides = sum(len(partie) for partie in identifiants)
        doublons_cle = valeurs_cle_valides - len(ids_uniques)

    ajouter_test(
        tests,
        f"{nom_fichier} — structure",
        True,
        f"{lignes:,} lignes; colonnes attendues présentes.",
    )

    for colonne, nombre in manquants.items():
        autorises = (
            commandes_premieres_sans_delai
            if nom_fichier == "orders.csv"
            and colonne == "days_since_prior_order"
            else 0
        )
        inattendus = nombre - autorises
        ajouter_test(
            tests,
            f"{nom_fichier} — valeurs manquantes ({colonne})",
            inattendus == 0,
            f"{nombre:,} manquantes; {autorises:,} attendues sur la première commande.",
        )

    if len(schema["cle"]) == 1 or schema["cle"] == ["order_id", "product_id"]:
        ajouter_test(
            tests,
            f"{nom_fichier} — unicité de la clé",
            doublons_cle == 0 and ordre_trie,
            (
                f"{doublons_cle:,} clé(s) dupliquée(s)."
                if ordre_trie
                else (
                    f"{doublons_cle:,} doublon(s) dans les morceaux; "
                    "l'ordre des commandes n'est pas croissant, le contrôle "
                    "global des clés composées n'est donc pas garanti."
                )
            ),
            avertissement=not ordre_trie,
        )

    for colonne, nombre in hors_plage.items():
        ajouter_test(
            tests,
            f"{nom_fichier} — plage de {colonne}",
            nombre == 0,
            f"{nombre:,} valeur(s) invalide(s) ou hors plage.",
        )

    for colonne, nombre in valeurs_invalides.items():
        if nombre:
            ajouter_test(
                tests,
                f"{nom_fichier} — format numérique de {colonne}",
                False,
                f"{nombre:,} valeur(s) non numériques.",
            )

    for nom_reference, nombre in references_absentes.items():
        ajouter_test(
            tests,
            f"{nom_fichier} — référence {nom_reference}",
            nombre == 0,
            f"{nombre:,} identifiant(s) absent(s) de la table de référence.",
        )

    corrections = []
    for colonne in schema["textes"]:
        nombre_espaces = corrections_espaces[colonne]
        nombre_vides = corrections_vides[colonne]
        if nombre_espaces or nombre_vides:
            corrections.append(
                f"- {colonne} : {nombre_espaces:,} espace(s) superflu(s) "
                f"retiré(s), {nombre_vides:,} chaîne(s) vide(s) normalisée(s)."
            )

    if nom_fichier == "orders.csv":
        ajouter_test(
            tests,
            "orders.csv — délai de première commande",
            delai_sur_premiere_commande == 0,
            (
                f"{commandes_premieres_sans_delai:,} délai(s) manquant(s) "
                "attendu(s) sur la première commande; "
                f"{commandes_non_premieres_sans_delai:,} manquant(s) "
                "sur une commande ultérieure."
            ),
        )

    return {
        "lignes": lignes,
        "ids": ids_uniques,
        "corrections": corrections,
        "manquants": manquants,
    }

## 5. Ex?cuter le traitement complet et produire le rapport

Le fichier `order_products__prior.csv` d?passe 32 millions de lignes ; le traitement peut durer plusieurs minutes. La progression s?affiche pendant l?analyse. Les valeurs manquantes du d?lai sur la premi?re commande sont attendues : il n?existe alors pas de commande pr?c?dente.

In [10]:
def main():
    fichiers_absents = [
        nom for nom in SCHEMAS if not (DOSSIER_RAW / nom).is_file()
    ]
    if fichiers_absents:
        raise FileNotFoundError(
            f"Fichiers bruts introuvables dans {DOSSIER_RAW} : "
            f"{', '.join(fichiers_absents)}"
        )

    taille_sorties = sum((DOSSIER_RAW / nom).stat().st_size for nom in SCHEMAS)
    espace_libre = shutil.disk_usage(DOSSIER_PROJET).free
    if espace_libre < taille_sorties:
        raise OSError(
            "Espace disque insuffisant pour créer les copies préparées : "
            f"{taille_sorties:,} octets nécessaires, "
            f"{espace_libre:,} octets disponibles."
        )

    DOSSIER_PREPARE.mkdir(exist_ok=True)
    DOSSIER_RAPPORTS.mkdir(exist_ok=True)
    tests = []
    resultats = {}

    resultats["aisles.csv"] = analyser_fichier(
        "aisles.csv", {}, tests
    )
    resultats["departments.csv"] = analyser_fichier(
        "departments.csv", {}, tests
    )
    references_produits = {
        "aisle_id": resultats["aisles.csv"]["ids"],
        "department_id": resultats["departments.csv"]["ids"],
    }
    resultats["products.csv"] = analyser_fichier(
        "products.csv", references_produits, tests
    )
    resultats["orders.csv"] = analyser_fichier(
        "orders.csv", {}, tests
    )

    references_commandes = {
        "order_id": resultats["orders.csv"]["ids"],
        "product_id": resultats["products.csv"]["ids"],
    }
    resultats["order_products__prior.csv"] = analyser_fichier(
        "order_products__prior.csv", references_commandes, tests
    )
    resultats["order_products__train.csv"] = analyser_fichier(
        "order_products__train.csv", references_commandes, tests
    )

    rapport = [
        "RAPPORT DE QUALITÉ, NETTOYAGE ET TESTS",
        "=" * 60,
        "Les fichiers data/raw/ ont été conservés sans modification.",
        f"Les copies préparées se trouvent dans : {DOSSIER_PREPARE}",
        "",
        "CORRECTIONS AUTOMATIQUES",
        "-" * 60,
        (
            "Seuls les espaces en début/fin des champs texte ont été retirés "
            "et les champs texte composés d'espaces ont été normalisés."
        ),
        (
            "Aucune ligne n'a été supprimée et aucune valeur numérique ou "
            "manquante n'a été imputée automatiquement."
        ),
    ]

    corrections_trouvees = False
    for nom, resultat in resultats.items():
        if resultat["corrections"]:
            corrections_trouvees = True
            rapport.append(f"\n{nom}")
            rapport.extend(resultat["corrections"])
    if not corrections_trouvees:
        rapport.append("Aucune correction de texte nécessaire.")

    rapport.extend(["", "RÉSULTATS DES TESTS", "-" * 60])
    rapport.extend(tests)

    nombre_echecs = sum("[ÉCHEC]" in test for test in tests)
    nombre_avertissements = sum("[AVERTISSEMENT]" in test for test in tests)
    rapport.extend(
        [
            "",
            "SYNTHÈSE",
            "-" * 60,
            f"Tests en échec : {nombre_echecs}",
            f"Avertissements : {nombre_avertissements}",
            (
                "Des anomalies nécessitent une correction ou une validation "
                "manuelle."
                if nombre_echecs or nombre_avertissements
                else "Les contrôles définis sont tous satisfaits."
            ),
        ]
    )

    chemin_rapport = DOSSIER_RAPPORTS / "rapport_qualite_donnees.txt"
    chemin_rapport.write_text("\n".join(rapport) + "\n", encoding="utf-8")
    print("\n".join(rapport))
    print(f"\nRapport enregistré : {chemin_rapport}")

## 6. Consulter le rapport et v?rifier une copie pr?par?e

Dans le rapport, `[OK]` indique un contr?le r?ussi et `[?CHEC]` un probl?me ? examiner. Les fichiers d?origine restent dans `data/raw/`.

In [11]:
chemin_rapport = DOSSIER_PREPARE / "rapport_qualite_donnees.txt"
rapport = chemin_rapport.read_text(encoding="utf-8")
print(rapport)

display(pd.read_csv(DOSSIER_PREPARE / "products.csv", nrows=5))

RAPPORT DE QUALITÉ, NETTOYAGE ET TESTS
Les fichiers data/raw/ ont été conservés sans modification.
Les copies préparées se trouvent dans : C:\Users\mouft\Desktop\data\data_m2\data\processed

CORRECTIONS AUTOMATIQUES
------------------------------------------------------------
Seuls les espaces en début/fin des champs texte ont été retirés et les champs texte composés d'espaces ont été normalisés.
Aucune ligne n'a été supprimée et aucune valeur numérique ou manquante n'a été imputée automatiquement.

products.csv
- product_name : 4 espace(s) superflu(s) retiré(s), 0 chaîne(s) vide(s) normalisée(s).

RÉSULTATS DES TESTS
------------------------------------------------------------
[OK] aisles.csv — structure : 134 lignes; colonnes attendues présentes.
[OK] aisles.csv — valeurs manquantes (aisle_id) : 0 manquantes; 0 attendues sur la première commande.
[OK] aisles.csv — valeurs manquantes (aisle) : 0 manquantes; 0 attendues sur la première commande.
[OK] aisles.csv — unicité de la clé : 0 

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## ? retenir

- Les six sources brutes sont conserv?es sans modification.
- Les copies pr?par?es et le rapport sont dans `data/processed/`.
- Les espaces superflus des champs texte sont retir?s.
- Les d?lais manquants de la premi?re commande d?un client sont conserv?s car ils sont normaux.
- Les ?tapes ult?rieures du projet (segmentation, associations ou classification) pourront ?tre pr?sent?es dans un notebook distinct.